In [49]:
import os

BASE_DIR = "/data8/enver/projects/conformal/dist_bench"
DATA_DIR = os.path.join(BASE_DIR, "data")
CODE_DIR = os.path.join(BASE_DIR, "code")
FC_DIR = os.path.join(BASE_DIR, 'models_save/fc')
UC_DIR = os.path.join(BASE_DIR, 'models_save/uc')

import sys

sys.path.append(CODE_DIR)

import os
import json
from collections import defaultdict

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import matplotlib.gridspec as grid_spec

from tqdm.auto import tqdm

from loader.generator import DataGenerator
from config import TSDataConfig, TaskConfig
from main_utils import _init_fc
from omegaconf import DictConfig

from models.forcast.darts import SimpleDartsModel
from models.forcast.forcast_service import ForcastService
from models.forcast.forcast_base import FCPredictionData
from models.uncertainty.uc_service import UncertaintyService
from models.uncertainty.dist_match.tree import DistMatchQRF
from models.uncertainty.dist_match.utils import match_ks_stat
from utils.calc_torch import calc_residuals

In [50]:
def matcher(x1, x2):
    return match_ks_stat(x1, x2) < 0.6


def get_qrf(path: str) -> DistMatchQRF:
    qrf = DistMatchQRF(
        alpha=0.1,
        n_quantile_bins=10,
        feature_dim=-1,
        matcher=matcher,
        match_mask=None,
        n_trees=10,
        bagging_ratio=0.9,
        verbose=False,
    )
    qrf.load_trees(path)
    return qrf

In [51]:
TASK_CONFIG = DictConfig(
    {
        "task_type": "PI",
        "alpha": 0.1,
        "data_splits": [0.6, 0.15, 0.25],
        "fc_estimator_mode": "single",
        "global_norm": False,
        "add_config": None,
    }
)


def load_dataset(data_config: TSDataConfig, task_config: TaskConfig = None):
    task_config = task_config or TaskConfig(**TASK_CONFIG)
    return DataGenerator.get_data(
        data_config=data_config,
        task_config=TASK_CONFIG,
        replace_base_dir=DATA_DIR,
        X_norm_param=None,
        Y_norm_param=None,
        hydro_static_norm_param=None,
    )


def load_forecast_service(
    fc_model=None, model_config=None, data_config=None, task_config=None
):
    if fc_model is None:
        fc_model = SimpleDartsModel
        model_config = dict(
            model="darts-forest", model_params={"lags": 50, "lags_past_covariates": 50}
        )

    task_config = task_config or TASK_CONFIG

    return ForcastService(
        int_fc_model=lambda: fc_model(**model_config),
        task_config=task_config,
        model_config=model_config,
        data_config=data_config,
        persist_dir=FC_DIR,
    )

In [52]:
def find_prototype(target, node):
    samples = node.get_values()[0]
    parent_split_values = [parent.get_split_value()[0] for parent in node.get_all_parents()]

    samples = [
        sample
        for sample in samples
        if not any(np.allclose(sample, parent) for parent in parent_split_values)
    ]

    min_dist = None
    best_sample = None
    for sample in samples:
        target_renorm = target
        dist = (abs(target_renorm - sample)).mean()
        if min_dist is None or dist < min_dist:
            min_dist = dist
            best_sample = sample

    return best_sample

In [53]:
def get_residuals(forcast_service: ForcastService, data, is_calib: bool = False):
    data = forcast_service.prepare(data, forcast_service._task_config.alpha)
    
    if is_calib:
        calib_data = UncertaintyService._map_to_calib_data(data)
        fc_result = forcast_service.predict(
            FCPredictionData(
                ts_id=calib_data.ts_id,
                X_past=calib_data.X_pre_calib,
                Y_past=calib_data.Y_pre_calib,
                X_step=calib_data.X_calib,
                step_offset=calib_data.step_offset,
            )
        )
        return calc_residuals(Y_hat=fc_result.point, Y=calib_data.Y_calib).numpy()

    fc_result = forcast_service.predict(
        FCPredictionData(
            ts_id=data.ts_id,
            X_past=data.X_calib,
            Y_past=data.Y_calib,
            X_step=data.X_test,
            step_offset=data.test_step,
        )
    )
    return calc_residuals(Y_hat=fc_result.point, Y=data.Y_test).numpy()

In [54]:
def get_data_prototypes(
    data,
    qrf,
    forcast_service,
    patch_len,
    start: int = 0,
    stop: int = 1000,
    step: int = 10,
    group_colors: list = None,
    n_samples: int = 4,
    is_calib: bool = True,
):
    group_colors = group_colors or ["#FFA332", "#9DC3FE"]
    ex_data = get_residuals(forcast_service, data, is_calib)
    ex_data = np.ravel(ex_data)

    tree = qrf.trees[0]

    node_idx_map = {node: idx for idx, node in enumerate(tree.leaf_nodes)}

    prototypes = {}
    prototype_samples = defaultdict(list)

    ex_data = ex_data[start:stop]
    proto_src_data = np.lib.stride_tricks.sliding_window_view(ex_data, patch_len)[
        ::step
    ]
    p_bar = tqdm(proto_src_data, leave=False)
    for cur_idx, patch in enumerate(p_bar):
        node = tree.predict_single_node(patch)
        node_idx = node_idx_map[node]
        prototype = find_prototype(patch, node)
        if node.parent.right != node:
            continue
        prototype = node.parent.get_split_value()[0]
        if node_idx not in prototypes:
            prototypes[node_idx] = prototype
        
        prototype_samples[node_idx].append(patch)

    n_groups = len(group_colors)

    max_prototypes = sorted(prototype_samples.items(), key=lambda x: len(x[1]), reverse=True)[:n_groups]
    max_prototypes = [proto_idx[0] for proto_idx in max_prototypes]

    fig, axes = plt.subplots(nrows=n_groups, ncols=n_samples + 1, figsize=(2 * (n_samples + 1), 2 * n_groups))

    for idx, proto_id in enumerate(max_prototypes):
        samples = prototype_samples[proto_id]
        sample_ids = np.random.choice(len(samples), n_samples)
        samples = np.array(samples)[sample_ids]

        color = group_colors[idx]
        axes[idx, 0].plot(prototypes[proto_id], color=color, linewidth=2)
        axes[idx, 0].tick_params(axis='x', labelsize=15)
        axes[idx, 0].tick_params(axis='y', labelsize=15)

        y_min = prototypes[proto_id].min()
        y_max = prototypes[proto_id].max()

        for i, sample in enumerate(samples):
            ax = axes[idx, i+1]
            ax.plot(sample, color=color, linewidth=2)
            y_min = min(y_min, sample.min())
            y_max = max(y_max, sample.max())

            ax.set_yticklabels([])

            # spines = ["top","right","left","bottom"]
            spines = ["right", "bottom"]
            for s in spines:
                ax.spines[s].set_visible(False)
            
            ax.tick_params(axis='x', labelsize=15)
        
        for ax in axes[idx]:
            ax.set_ylim(y_min, y_max)

    l_offset = 0.045
    r_offset = 0.01
    sep_color = "#b7b7b7"
    v_sep_pos = l_offset + (idx) / (n_samples + 1) * (1 - l_offset - r_offset)
    v_sep_line = plt.Line2D([v_sep_pos, v_sep_pos], [0.025, 0.975], transform=fig.transFigure, color=sep_color, linewidth=2)
    fig.add_artist(v_sep_line)

    fig.legend()
    fig.tight_layout()

    return fig

In [63]:
def get_data_path(
    data,
    qrf,
    forcast_service,
    patch_len,
    start: int = 0,
    stop: int = 1000,
    step: int = 10,
    color: str = None,
    n_samples: int = 4,
    is_calib: bool = True,
):
    color = color or "#9DC3FE"
    ex_data = get_residuals(forcast_service, data, is_calib)
    ex_data = np.ravel(ex_data)

    tree = qrf.trees[0]

    path_patch_pairs = []

    ex_data = ex_data[start:stop]
    proto_src_data = np.lib.stride_tricks.sliding_window_view(ex_data, patch_len)[
        ::step
    ]
    p_bar = tqdm(proto_src_data, leave=False)
    for cur_idx, patch in enumerate(p_bar):
        node = tree.predict_single_node(patch)
        path = []
        while node.parent is not None:
            cur_node = node
            node = node.parent
            if node.right != cur_node:
                continue
            path.append(cur_node)
        path_patch_pairs.append((path, patch))

    path_figs = []
    for path_idx, (path, patch) in enumerate(path_patch_pairs):
        node_figs = []
        for node_idx, node in enumerate(path):
            values_tuple = node.get_values()
            if values_tuple is None:
                continue
            xs = values_tuple[0]
            values = np.sort(xs.ravel())
            ecdf_y = np.arange(1, len(values) + 1) / len(values)

            fig, (ax_ts, ax_cdf) = plt.subplots(1, 2, figsize=(12, 4))

            ax_ts.plot(patch, color=color, linewidth=2)
            ax_ts.set_xlabel("Time step")
            ax_ts.set_ylabel("Residual")
            ax_ts.set_title("Original patch")

            ax_cdf.step(values, ecdf_y, color=color, linewidth=2, where='post')
            ax_cdf.set_xlabel("Residual value")
            ax_cdf.set_ylabel("eCDF")
            ax_cdf.set_title("Node eCDF")

            fig.suptitle(f"Path {path_idx}, Node {node_idx}")
            plt.tight_layout()

            node_figs.append(fig)
        path_figs.append(node_figs)

    return path_figs

In [ ]:
PATCH_LEN = 100
TREE_DIR = os.path.join(BASE_DIR, "models_save/uc/")
DATA_MAP = {
    "Elec": (
        "qrf_ks_stat<0.1_error_normal_electricelectricity-normalized_darts-forest|100.pkl",
        "electric",
        ["/some_base_dir/data/enbPI/electricity-normalized.csv"],
        0,
    ),
}

tree_figs = {}

for data_type, (qrf_path, datatype, data_paths, file_idx) in DATA_MAP.items():
    qrf = get_qrf(os.path.join(TREE_DIR, qrf_path))
    data_config = DictConfig(
        {"dataset_type": datatype, "paths": data_paths, "add_config": None}
    )
    forcast_service = load_forecast_service(data_config=data_config)
    datasets = load_dataset(TSDataConfig(**data_config))
    # get_data_prototypes(datasets[file_idx], qrf, forcast_service, PATCH_LEN)
    tree_figs[data_type] = get_data_path(datasets[file_idx], qrf, forcast_service, PATCH_LEN)

print(tree_figs)
for data_type, data_figs in tree_figs.items():
    for sample_figs in data_figs:
        for fig in sample_figs:
            fig.tight_layout()
            fig.show()

  0%|          | 0/42 [00:00<?, ?it/s]